# Beginner E2: classify accents with frozen Whisper embeddings

This is a small, step-by-step learning version of the E2 experiment (Transfer Learning). A pretrained **Whisper encoder** turns each recording into an embedding, and a small **linear classifier head** learns from those embeddings. Whisper stays frozen: we do not train the large speech model.

By the end, you will know how to:

1. load and normalize audio;
2. extract and cache frozen Whisper encoder embeddings;
3. keep speakers separate between training and testing;
4. train a small classifier head; and
5. read F1, balanced accuracy and a confusion matrix.

> This experiment distinguishes the two supplied speech datasets. It must not be used to infer ethnicity, nationality or identity.

## Before you start

In Colab, choose **Runtime → Change runtime type → T4 GPU**. A GPU makes embedding extraction much faster. The classifier itself is small and runs on the CPU.

In [ ]:
# Install the libraries used in this notebook.
%pip -q install "transformers>=4.46" "librosa>=0.10.2" scikit-learn seaborn joblib soundfile

## 1. Arrange the audio by class and speaker

Use the same `Singlish/data/audio` folder in Google Drive as the E1 notebooks. This lets E1 and E2 evaluate the same recordings without copying the audio:

```text
Singlish/
├── data/
│   └── audio/
│       ├── singapore/
│       │   ├── speaker_001/
│       │   │   ├── clip_01.wav
│       │   │   └── clip_02.wav
│       │   └── speaker_002/
│       │       └── clip_01.wav
│       └── non_singapore/
│           ├── speaker_101/
│           │   └── clip_01.mp3
│           └── speaker_102/
│               └── clip_01.mp3
└── outputs/
    └── e2/                 # Created automatically
```

Only `data/audio` needs to exist before the run. The notebook automatically creates `Singlish/outputs/e2` for its results.

The first directory below a class must identify the speaker. Each class needs at least two speakers; five or more per class is preferable. Speaker folders prevent one person's clips from appearing in both training and test data.

In [ ]:
from pathlib import Path

# Google Colab: mount Drive.
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

# Configure only this path. E1 and E2 share data/audio but have separate outputs.
PROJECT_ROOT = Path('/content/drive/MyDrive/Singlish')
DATA_ROOT = PROJECT_ROOT / 'data' / 'audio'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'e2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
MAX_CLIPS_PER_CLASS = 100  # Set None to use all audio.
SAMPLE_RATE = 16_000
MAX_SECONDS = 6
MODEL_NAME = 'openai/whisper-base.en'  # Faster than small.en for learning.
BATCH_SIZE = 8
REBUILD_CACHE = False  # Change to True after adding/removing audio.

print('Project folder:', PROJECT_ROOT)
print('Looking for audio in:', DATA_ROOT)
print('Saving results to:', OUTPUT_DIR)

In [ ]:
import json
from collections import Counter

import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from transformers import WhisperFeatureExtractor, WhisperModel

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch device:', DEVICE)
if DEVICE.type != 'cuda':
    print('Warning: no GPU detected. The notebook will work, but embedding extraction will be slow.')

## 2. Find the recordings

Singapore speech receives label `1` and non-Singapore speech receives label `0`. The class name is prefixed to each speaker ID so an identical folder name in the two datasets cannot create a false match.

In [ ]:
AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}
CLASS_LABELS = {'non_singapore': 0, 'singapore': 1}


def find_recordings(data_root):
    rows = []
    for class_name, label in CLASS_LABELS.items():
        class_dir = data_root / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f'Missing class directory: {class_dir}')
        for path in sorted(class_dir.rglob('*')):
            if not path.is_file() or path.suffix.lower() not in AUDIO_EXTENSIONS:
                continue
            relative_parts = path.relative_to(class_dir).parts
            if len(relative_parts) < 2:
                raise ValueError(f'Put this clip inside a speaker folder: {path}')
            speaker_id = relative_parts[0]
            group_id = f'{class_name}::{speaker_id}'
            rows.append((path, label, group_id))
    if not rows:
        raise RuntimeError(f'No supported audio files found below {data_root}')
    return rows


def select_recordings(rows, max_clips_per_class, seed):
    if max_clips_per_class is None:
        return rows
    if not isinstance(max_clips_per_class, int) or isinstance(max_clips_per_class, bool) or max_clips_per_class < 1:
        raise ValueError('MAX_CLIPS_PER_CLASS must be None or a positive integer')
    import random
    rng = random.Random(seed)
    selected = []
    for label in CLASS_LABELS.values():
        by_speaker = {}
        for row in rows:
            if row[1] == label:
                by_speaker.setdefault(row[2], []).append(row)
        speakers = sorted(by_speaker)
        rng.shuffle(speakers)
        for speaker in speakers:
            rng.shuffle(by_speaker[speaker])
        class_count = 0
        while class_count < max_clips_per_class:
            added = False
            for speaker in speakers:
                if by_speaker[speaker]:
                    selected.append(by_speaker[speaker].pop())
                    added = True
                    class_count += 1
                    if class_count == max_clips_per_class:
                        break
            if not added:
                break
    return sorted(selected, key=lambda row: str(row[0]))


recordings = select_recordings(find_recordings(DATA_ROOT), MAX_CLIPS_PER_CLASS, RANDOM_SEED)
print('Recordings:', len(recordings))
print('Labels:', Counter(label for _, label, _ in recordings))
print('Speakers per class:')
for class_name, label in CLASS_LABELS.items():
    speakers = {group for _, item_label, group in recordings if item_label == label}
    print(f'  {class_name}: {len(speakers)}')

## 3. Convert audio into Whisper embeddings

Whisper first converts each waveform into a log-Mel spectrogram. Its pretrained encoder then produces a sequence of learned representations. We average that sequence over time to obtain one fixed-length embedding per clip.

`model.eval()` and `torch.inference_mode()` disable training behavior and gradients. The Whisper weights therefore remain frozen. Short clips are padded by Whisper's feature extractor, which keeps every batch the expected size.

In [ ]:
def load_and_prepare_audio(path):
    y, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    y, _ = librosa.effects.trim(y, top_db=30)
    if y.size == 0:
        raise ValueError(f'No audible signal: {path}')
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak
    y = librosa.util.fix_length(y, size=max(len(y), SAMPLE_RATE))
    return y[:SAMPLE_RATE * MAX_SECONDS].astype(np.float32)


def load_frozen_encoder():
    feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
    whisper = WhisperModel.from_pretrained(MODEL_NAME).to(DEVICE)
    encoder = whisper.get_encoder().eval()
    for parameter in encoder.parameters():
        parameter.requires_grad = False
    return feature_extractor, encoder


@torch.inference_mode()
def encode_waveforms(waveforms, feature_extractor, encoder):
    batch = feature_extractor(
        waveforms, sampling_rate=SAMPLE_RATE, return_tensors='pt'
    ).input_features.to(DEVICE)
    hidden = encoder(input_features=batch).last_hidden_state
    return hidden.mean(dim=1).float().cpu().numpy()

### Extract once, then cache

Whisper is the expensive step. The cache lets later runs train the classifier immediately. If the model name or file list changes, the notebook asks you to rebuild rather than silently using stale embeddings. Skipped unreadable files are recorded in the cache through the final kept file list.

In [ ]:
cache_suffix = '' if MAX_CLIPS_PER_CLASS is None else f'_max{MAX_CLIPS_PER_CLASS}'
EMBEDDING_CACHE = OUTPUT_DIR / f'beginner_whisper_embeddings{cache_suffix}.npz'
current_paths = np.asarray([str(path) for path, _, _ in recordings])

if EMBEDDING_CACHE.exists() and not REBUILD_CACHE:
    with np.load(EMBEDDING_CACHE, allow_pickle=False) as cache:
        cached_model = str(cache['model_name'])
        cached_paths = cache['paths'].astype(str)
        discovered_paths = cache['discovered_paths'].astype(str)
        cached_sample_rate = int(cache['sample_rate'])
        cached_max_seconds = int(cache['max_seconds'])
        cache_matches = (
            cached_model == MODEL_NAME
            and cached_sample_rate == SAMPLE_RATE
            and cached_max_seconds == MAX_SECONDS
            and np.array_equal(discovered_paths, current_paths)
        )
        if not cache_matches:
            raise RuntimeError(
                'The audio list or Whisper model changed. Set REBUILD_CACHE = True and rerun.'
            )
        X = cache['embeddings'].astype(np.float32)
        y = cache['labels'].astype(np.int64)
        groups = cache['groups'].astype(str)
        paths = cached_paths
    print('Loaded cached embeddings:', EMBEDDING_CACHE)
else:
    feature_extractor, encoder = load_frozen_encoder()
    embedding_batches = []
    kept_rows = []
    for start in range(0, len(recordings), BATCH_SIZE):
        batch_rows = recordings[start:start + BATCH_SIZE]
        waveforms, valid_rows = [], []
        for row in batch_rows:
            try:
                waveforms.append(load_and_prepare_audio(row[0]))
                valid_rows.append(row)
            except Exception as error:
                print(f'Skipping {row[0].name}: {error}')
        if waveforms:
            embedding_batches.append(encode_waveforms(waveforms, feature_extractor, encoder))
            kept_rows.extend(valid_rows)
        print(f'Processed {min(start + BATCH_SIZE, len(recordings))}/{len(recordings)}')

    X = np.vstack(embedding_batches).astype(np.float32)
    y = np.asarray([label for _, label, _ in kept_rows], dtype=np.int64)
    groups = np.asarray([group for _, _, group in kept_rows])
    paths = np.asarray([str(path) for path, _, _ in kept_rows])
    np.savez_compressed(
        EMBEDDING_CACHE, embeddings=X, labels=y, groups=groups, paths=paths,
        discovered_paths=current_paths,
        model_name=np.asarray(MODEL_NAME), sample_rate=np.asarray(SAMPLE_RATE),
        max_seconds=np.asarray(MAX_SECONDS),
    )
    del encoder
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    print('Saved embedding cache:', EMBEDDING_CACHE)

assert len(X) == len(y) == len(groups) == len(paths)
assert np.isfinite(X).all()
print('Embedding matrix shape:', X.shape)
print('One row = one recording; one column = one Whisper representation value.')

## 4. Make a speaker-disjoint split

This is the same rule as E1: no speaker may occur on both sides. With at least five speakers per class, the first fold is approximately an 80/20 training/test split. With fewer speakers, the notebook automatically uses fewer folds.

> For the final project, create train/validation/test speaker splits and use validation—not test—for model choices. This compact notebook keeps only train/test to make the basic E2 workflow easy to follow.

In [ ]:
speaker_counts = {
    label: len(set(groups[y == label]))
    for label in sorted(np.unique(y))
}
if set(speaker_counts) != {0, 1}:
    raise ValueError('Both class labels must remain after embedding extraction')
n_splits = min(5, min(speaker_counts.values()))
if n_splits < 2:
    raise ValueError('Each class needs audio from at least two speakers')

splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(X, y, groups))
train_speakers = set(groups[train_idx])
test_speakers = set(groups[test_idx])
assert not (train_speakers & test_speakers)
assert set(np.unique(y[train_idx])) == {0, 1}
assert set(np.unique(y[test_idx])) == {0, 1}

print('Training clips:', len(train_idx), '| labels:', Counter(y[train_idx]))
print('Test clips:    ', len(test_idx), '| labels:', Counter(y[test_idx]))
print('Speaker overlap:', len(train_speakers & test_speakers))

## 5. Train the small classifier head

Whisper has already done the deep representation learning. We now fit one linear decision layer using logistic regression. This is a small classifier head: it learns only how to combine the frozen embedding values into a Singapore-class probability.

`StandardScaler` and the classifier are fitted only on training speakers, preventing information from the test set from leaking into scaling.

In [ ]:
classifier = Pipeline([
    ('scale', StandardScaler()),
    ('head', LogisticRegression(
        C=1.0, class_weight='balanced', max_iter=2000,
        random_state=RANDOM_SEED,
    )),
])
classifier.fit(X[train_idx], y[train_idx])
print('Classifier-head training complete. Whisper remained frozen.')

## 6. Evaluate unseen speakers

Balanced accuracy gives both classes equal importance, macro F1 gives both class-specific F1 scores equal weight, and the confusion matrix shows the exact error types. Compare these results with E1 only when both experiments use the same recordings and speaker split.

In [ ]:
predictions = classifier.predict(X[test_idx])
matrix = confusion_matrix(y[test_idx], predictions, labels=[0, 1])
metrics = {
    'accuracy': float(accuracy_score(y[test_idx], predictions)),
    'balanced_accuracy': float(balanced_accuracy_score(y[test_idx], predictions)),
    'macro_f1': float(f1_score(y[test_idx], predictions, average='macro')),
    'confusion_matrix': matrix.tolist(),
}
print(json.dumps(metrics, indent=2))
print('\nDetailed report:\n')
print(classification_report(
    y[test_idx], predictions, labels=[0, 1],
    target_names=['Non-Singapore', 'Singapore'], digits=3, zero_division=0,
))

In [ ]:
sns.set_theme(style='white')
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    matrix, annot=True, fmt='d', cmap='Purples', cbar=False, ax=ax,
    xticklabels=['Non-SG', 'Singapore'], yticklabels=['Non-SG', 'Singapore'],
)
ax.set(title='Beginner E2 Whisper test', xlabel='Predicted label', ylabel='True label')
fig.tight_layout()
confusion_path = OUTPUT_DIR / 'confusion_matrix.png'
fig.savefig(confusion_path, dpi=180, bbox_inches='tight')
plt.show()

## 7. Save the experiment

`Singlish/outputs/e2` will contain the embedding cache, fitted classifier head, exact split, report and confusion-matrix image. The classifier file does **not** contain Whisper itself; inference still needs the same Hugging Face model.

In [ ]:
classifier_path = OUTPUT_DIR / 'beginner_whisper_linear_head.joblib'
split_path = OUTPUT_DIR / 'beginner_whisper_split.npz'
report_path = OUTPUT_DIR / 'beginner_whisper_report.json'

joblib.dump(classifier, classifier_path)
np.savez_compressed(
    split_path, paths=paths, labels=y, groups=groups,
    train_indices=train_idx, test_indices=test_idx,
)
report = {
    'embedding_model': MODEL_NAME,
    'pooling': 'mean over encoder time steps',
    'classifier': 'StandardScaler + LogisticRegression',
    'label_meaning': {'0': 'non_singapore', '1': 'singapore'},
    'embedding_size': int(X.shape[1]),
    'train_samples': int(len(train_idx)),
    'test_samples': int(len(test_idx)),
    'train_speakers': int(len(train_speakers)),
    'test_speakers': int(len(test_speakers)),
    'speaker_overlap': 0,
    'metrics': metrics,
    'caution': 'Corpus and recording differences may be learned instead of accent.',
}
report_path.write_text(json.dumps(report, indent=2) + '\n')

print('Saved:')
for path in (EMBEDDING_CACHE, classifier_path, split_path, report_path, confusion_path):
    print(' -', path)

## 8. Try one recording

This helper repeats the same preprocessing, frozen encoder and classifier head for one file. Loading Whisper can take a moment the first time. Keep the returned value framed as a model score—not a definitive statement about a person.

In [ ]:
def predict_recording(audio_path):
    feature_extractor, encoder = load_frozen_encoder()
    waveform = load_and_prepare_audio(Path(audio_path))
    embedding = encode_waveforms([waveform], feature_extractor, encoder)
    probability = float(classifier.predict_proba(embedding)[0, 1])
    predicted_name = 'Singapore' if probability >= 0.5 else 'Non-Singapore'
    print(f'Prediction: {predicted_name}')
    print(f'Singapore-class probability: {probability:.1%}')
    return probability

# Example — replace this with one of your audio paths:
# predict_recording(DATA_ROOT / 'singapore' / 'speaker_001' / 'clip_01.wav')

## What to try next

Once this notebook makes sense:

1. use exactly the same clips and speaker split as E1 for a fair comparison;
2. increase the number and variety of speakers;
3. try `openai/whisper-small.en` by changing `MODEL_NAME` and rebuilding the cache;
4. add a speaker-disjoint validation split before tuning the classifier; and
5. inspect whether corpus, microphone, duration or prompt differences explain unusually high scores.

Do not repeatedly change settings after looking at the test score. Keep the test speakers untouched until the final evaluation.